In [ ]:
import os
from pathlib import Path
import time
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import cumulative_trapezoid
import scipy.sparse as sp
import scipy.sparse.linalg as spla

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset

from numpy.polynomial.chebyshev import chebval
import json

# =========================
def set_seed(seed=2026):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    rng = np.random.default_rng(seed)

    return rng

rng = set_seed(2026)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

fig_dir = Path("figs_uhat0")
fig_dir.mkdir(exist_ok=True)
torch.set_default_device("cpu")

In [ ]:
def set_size(width, fraction=1, subplots=(1, 1), height_add=0):
    """Set figure dimensions to avoid scaling in LaTeX.

    Parameters
    ----------
    width: float or string
            Document width in points, or string of predined document type
    fraction: float, optional
            Fraction of the width which you wish the figure to occupy
    subplots: array-like, optional
            The number of rows and columns of subplots.
    Returns
    -------
    fig_dim: tuple
            Dimensions of figure in inches
    """
    if width == 'thesis':
        width_pt = 426.79135
    elif width == 'beamer':
        width_pt = 307.28987
    else:
        width_pt = width

    # Width of figure (in pts)
    fig_width_pt = width_pt * fraction
    # Convert from pt to inches
    inches_per_pt = 1 / 72.27

    # Golden ratio to set aesthetic figure height
    #golden_ratio = (5**.5 - 1) / 2
    golden_ratio = 0.28

    
    fig_width_in = fig_width_pt * inches_per_pt
    fig_height_in = height_add + fig_width_in * golden_ratio * (subplots[0] / subplots[1])

    return (fig_width_in, fig_height_in)

tex_fonts = {
    # Use LaTeX to write all text
    "text.usetex": True,
    "font.family": "times",
    # Use 10pt font in plots, to match 10pt font in document
    "axes.labelsize": 14,
    "font.size": 14,
    # Make the legend/label fonts a little smaller
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12
}

plt.rcParams.update(tex_fonts)
plt.rcParams["axes.unicode_minus"] = False



def style_3d_axes(ax3d):
    for axis in [ax3d.xaxis, ax3d.yaxis, ax3d.zaxis]:
        axis._axinfo['axisline']['linewidth'] = 1
        axis._axinfo['axisline']['color'] = "black"
        axis._axinfo['grid']['linewidth'] = 0.2
        axis._axinfo['grid']['linestyle'] = "--"
        axis._axinfo['grid']['color'] = "#d1d1d1"
    # 纯白背景
    ax3d.xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
    ax3d.yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
    ax3d.zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))


def plot_3d_response_draw_style(
    u_data,
    spatial,
    temporal,
    zlabel=r"$u$",
    title=None,
    filename=None,
    rstride=30,
    cstride=4,
    figsize=None,
    zlim=None,
):
    """
    Plot u(x,t) in the same 3D style as draw.ipynb.

    u_data:
        shape [nt, Nx]
    spatial:
        shape [Nx]
    temporal:
        shape [nt]
    """

    meshx, mesht = np.meshgrid(spatial, temporal)

    if figsize is None:
        figsize = set_size(516 * 3, fraction=0.99, subplots=(1, 1), height_add=0.1)

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection="3d", computed_zorder=False)

    ax.plot_surface(
        meshx,
        mesht,
        u_data,
        edgecolor="black",
        lw=0.2,
        rstride=rstride,
        cstride=cstride,
        alpha=1,
        color="white",
        shade=False,
        rasterized=False,
    )

    # Right boundary trajectory x=1
    x_right = np.ones_like(temporal)
    vals_right = u_data[:, -1]

    ax.plot(
        x_right,
        temporal,
        vals_right,
        color="red",
        lw=1.3,
        antialiased=False,
    )

    style_3d_axes(ax)

    ax.view_init(10, 15)
    if zlim is not None:
        ax.set_zlim(zlim)
        
    ax.set_xlabel("x")
    ax.set_ylabel(r"$t$")
    ax.set_zlabel(zlabel)

    ax.set_xticks([0, 0.5, 1])
    ax.zaxis.set_rotate_label(False)
    ax.zaxis.label.set_rotation(90)
    # ax.zaxis.get_offset_text().set_rotation(0)
    # ax.zaxis.get_offset_text().set_horizontalalignment("left")
    # ax.zaxis.get_offset_text().set_verticalalignment("bottom")

    if title is not None:
        ax.set_title(title)

    plt.tight_layout()

    if filename is not None:
        fig.savefig(filename, dpi=300, bbox_inches="tight", pad_inches=0.06)

    plt.show()

    return fig, ax


def plot_2d_draw_style(
    x,
    ys,
    labels=None,
    linestyles=None,
    xlabel="time",
    ylabel=None,
    filename=None,
    grid=True,
):
    """
    2D line plot in the same style as draw.ipynb.
    """

    plt.figure(figsize=(1000 / 300, 618 / 300), dpi=300)

    if not isinstance(ys, (list, tuple)):
        ys = [ys]

    if labels is None:
        labels = [None] * len(ys)

    if linestyles is None:
        linestyles = ["-"] * len(ys)

    for y, label, ls in zip(ys, labels, linestyles):
        plt.plot(
            x,
            y,
            ls,
            label=label,
            linewidth=1.5,
        )

    plt.xlabel(xlabel, fontsize=8)

    if ylabel is not None:
        plt.ylabel(ylabel, fontsize=8)

    if any(label is not None for label in labels):
        plt.legend(fontsize=10)

    if grid:
        plt.grid(True)

    plt.tight_layout()

    if filename is not None:
        plt.savefig(filename, dpi=300, bbox_inches="tight")

    plt.show()

from mpl_toolkits.axes_grid1.inset_locator import Rectangle, inset_axes, mark_inset
import numpy as np
import matplotlib.pyplot as plt


def plot_2d_draw_style_with_inset(
    x,
    ys,
    labels=None,
    linestyles=None,
    ylim=None,      
    xlabel="time",
    ylabel=None,
    filename=None,
    grid=True,
    yscale="linear",
    legend_loc="best",
    legend_fontsize=6,
    zoom_xlim=None,
    zoom_ylim=None,
    inset_loc="upper right",
    inset_width="40%",
    inset_height="40%",
    inset_bbox=None,
):
    """
    2D line plot with an inset zoom region.

    zoom_xlim:
        tuple, for example (0.05, 0.30)
    zoom_ylim:
        tuple, for example (0.8, 1.2). If None, it is determined automatically.
    """

    fig, ax = plt.subplots(figsize=(1000 / 300, 618 / 300), dpi=300)

    if not isinstance(ys, (list, tuple)):
        ys = [ys]

    n_curves = len(ys)

    if labels is None:
        labels = [None] * n_curves

    if linestyles is None:
        linestyles = ["-", "--", ":", "-."][:n_curves]

    if len(labels) != n_curves:
        raise ValueError("labels and ys must have the same length.")

    if len(linestyles) != n_curves:
        raise ValueError("linestyles and ys must have the same length.")

    # Main plot
    for y, label, ls in zip(ys, labels, linestyles):
        ax.plot(
            x,
            y,
            ls,
            label=label,
            linewidth=1.5,
        )

    ax.set_xlabel(xlabel, fontsize=9)

    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=9)
    
    if ylim is not None:
        ax.set_ylim(ylim)

    ax.set_yscale(yscale)

    if grid:
        ax.grid(True)

    if any(label is not None for label in labels):
        ax.legend(fontsize=legend_fontsize, loc=legend_loc)
    
    ax.tick_params(axis="both", labelsize=8)

    # Inset zoom
    if zoom_xlim is not None:
        if inset_bbox is not None:
            axins = ax.inset_axes(inset_bbox)
        else:
            axins = inset_axes(ax, width=inset_width,height=inset_height,loc=inset_loc,borderpad=1.0,)

        for y, label, ls in zip(ys, labels, linestyles):
            axins.plot(
                x,
                y,
                ls,
                linewidth=1.2,
            )

        axins.set_xlim(zoom_xlim)

        if zoom_ylim is None:
            mask = (x >= zoom_xlim[0]) & (x <= zoom_xlim[1])

            y_min = min(np.min(np.asarray(y)[mask]) for y in ys)
            y_max = max(np.max(np.asarray(y)[mask]) for y in ys)

            margin = 0.08 * (y_max - y_min + 1e-12)
            zoom_ylim = (y_min - margin, y_max + margin)

        axins.set_ylim(zoom_ylim)

        axins.tick_params(axis="both", labelsize=7)

        if grid:
            axins.grid(True)

        # Draw rectangle and connectors
        # mark_inset(
        #     ax,
        #     axins,
        #     loc1=mark_loc1,
        #     loc2=mark_loc2,
        #     fc="none",
        #     ec="0.3",
        #     linewidth=0.8,
        # )
        # Draw only the zoom rectangle on the main axes, without connector lines.
        rect = Rectangle((zoom_xlim[0], zoom_ylim[0]),
                         zoom_xlim[1] - zoom_xlim[0],
                         zoom_ylim[1] - zoom_ylim[0],
                         fill=False, edgecolor="0.3", linewidth=0.8, linestyle="--",)

        ax.add_patch(rect)

    plt.tight_layout()

    if filename is not None:
        fig.savefig(filename, dpi=300, bbox_inches="tight")

    plt.show()

    return fig, ax

In [ ]:
# Part1 Define reaction coefficients, integral weights, and initial values.

def make_lam_fn(beta):
    """
    lambda(x)=50*cos(beta*arccos(x)).
    x should be in [0,1].
    """
    def lam(x):
        x = np.asarray(x)
        x_clip = np.clip(x, 0.0, 1.0)
        return 50.0 * np.cos(beta * np.arccos(x_clip))
    return lam


def get_simpson_coeff(spatial):
    """
    Simpson quadrature weights on [0,1].
    If the number of intervals is not even, use trapezoidal weights.
    """
    spatial = np.asarray(spatial, dtype=float)
    N = len(spatial)
    dx = spatial[1] - spatial[0]

    if (N - 1) % 2 != 0:
        w = np.ones(N) * dx
        w[0] *= 0.5
        w[-1] *= 0.5
        return w

    w = np.ones(N)
    w[1:-1:2] = 4.0
    w[2:-1:2] = 2.0
    w *= dx / 3.0
    return w


def make_initial_profile(
    spatial,
    init_type="sin",
    amp=1.0,
    rng=None,
    n_modes=8,
    decay_power=2.0,
    normalize=True,
):
    """
    Generate an initial state satisfying u(0)=u(1)=0.

    Parameters
    ----------
    spatial : array_like
        Spatial grid on [0, 1].
    init_type : str
        Type of initial condition:
        - "zero"
        - "sin"
        - "sin2"
        - "mixed"
        - "random_sine"
        - "cheb_single"
        - "cheb_random"
    amp : float
        Amplitude of the initial profile.
    rng : np.random.Generator or None
        Random number generator. If None, a default generator is used.
    n_modes : int
        Number of modes used in random expansions.
    decay_power : float
        Decay rate for high-order coefficients.
    normalize : bool
        Whether to normalize the generated profile before multiplying by amp.

    Returns
    -------
    u0 : ndarray
        Initial profile satisfying u0(0)=u0(1)=0.
    """

    x = np.asarray(spatial, dtype=float)

    if rng is None:
        rng = np.random.default_rng()

    if init_type == "zero":
        profile = np.zeros_like(x)

    elif init_type == "sin":
        profile = np.sin(np.pi * x)

    elif init_type == "sin2":
        profile = np.sin(2.0 * np.pi * x)

    elif init_type == "mixed":
        profile = (
            0.8 * np.sin(np.pi * x)
            + 0.2 * np.sin(2.0 * np.pi * x)
        )

    elif init_type == "random_sine":
        profile = np.zeros_like(x)

        for j in range(1, n_modes + 1):
            coeff = rng.uniform(-1.0, 1.0) / (j ** decay_power)
            profile += coeff * np.sin(j * np.pi * x)

    elif init_type == "cheb_single":
        beta0 = rng.uniform(1.0, 10.0)
        x_clip = np.clip(x, 0.0, 1.0)

        profile = np.cos(beta0 * np.arccos(x_clip))

        # Enforce u0(0)=u0(1)=0.
        profile = x * (1.0 - x) * profile

    elif init_type == "cheb_random":
        xi = 2.0 * x - 1.0

        coeffs = rng.uniform(-1.0, 1.0, size=n_modes + 1)

        decay = np.array([
            (j + 1) ** (-decay_power)
            for j in range(n_modes + 1)
        ])

        coeffs = coeffs * decay

        profile = chebval(xi, coeffs)

        # Enforce u0(0)=u0(1)=0.
        profile = x * (1.0 - x) * profile

    else:
        raise ValueError(f"Unknown init_type: {init_type}")

    if normalize:
        max_abs = np.max(np.abs(profile))
        if max_abs > 1e-12:
            profile = profile / max_abs

    return amp * profile

# Some Denormalization Functions and
def denormalize_U(U_norm, stats):
    return U_norm * stats["U_std"] + stats["U_mean"]


def denormalize_p(p_norm, stats):
    return p_norm * stats["p_std"] + stats["p_mean"]


def denormalize_lam(lam_norm, stats):
    return lam_norm * stats["lam_std"] + stats["lam_mean"]



# Part2 Define functions to solve the controller kernel k(x,y) and observer kernel g(x,y) using Goursat marching in characteristic coordinates.
def solve_k(lambda_func, spatial, return_Kxy=False, check=False):
    """
    Solve the controller kernel k(x,y) on 0 <= y <= x <= 1.

    Kernel equation:
        k_xx(x,y) - k_yy(x,y) = lambda(y) k(x,y)

    Boundary conditions:
        k(x,0) = 0
        k(x,x) = -1/2 int_0^x lambda(s) ds

    Characteristic coordinates:
        r = x + y
        s = x - y
    """

    spatial = np.asarray(spatial, dtype=float)
    N = len(spatial)
    dx = spatial[1] - spatial[0]

    if not np.allclose(np.diff(spatial), dx):
        raise ValueError("spatial must be a uniform grid.")

    Ns = N
    Nr = 2 * N - 1

    s_grid = np.linspace(0.0, 1.0, Ns)
    r_grid = np.linspace(0.0, 2.0, Nr)
    h = s_grid[1] - s_grid[0]

    ii = np.arange(Nr)[:, None]
    jj = np.arange(Ns)[None, :]
    valid = (ii >= jj) & (ii <= (Nr - 1 - jj))

    Krs = np.full((Nr, Ns), np.nan, dtype=np.float64)

    # Boundary s=0: k(x,x)
    x_int = spatial
    lam_int = np.asarray(lambda_func(x_int), dtype=np.float64)

    F = -0.5 * cumulative_trapezoid(lam_int, x_int, initial=0.0)
    Krs[:, 0] = np.interp(r_grid / 2.0, x_int, F)

    # Boundary r=s: k(x,0)=0
    for j in range(Ns):
        Krs[j, j] = 0.0

    # Goursat marching
    for i in range(Nr - 1):
        for j in range(Ns - 1):

            if not (
                valid[i, j]
                and valid[i + 1, j]
                and valid[i, j + 1]
                and valid[i + 1, j + 1]
            ):
                continue

            if (
                np.isnan(Krs[i + 1, j])
                or np.isnan(Krs[i, j + 1])
                or np.isnan(Krs[i, j])
            ):
                continue

            r_c = 0.5 * (r_grid[i] + r_grid[i + 1])
            s_c = 0.5 * (s_grid[j] + s_grid[j + 1])
            y_c = 0.5 * (r_c - s_c)

            a_c = 0.25 * float(lambda_func(y_c))
            q = (h * h * a_c) / 4.0

            SW = Krs[i, j]
            SE = Krs[i + 1, j]
            NW = Krs[i, j + 1]

            # Semi-implicit midpoint update for NE
            # NE - SE - NW + SW = h^2 a_c * (SW+SE+NW+NE)/4
            denom = 1.0 - q
            NE = (SE + NW - SW + q * (SW + SE + NW)) / denom

            Krs[i + 1, j + 1] = NE

    # Extract k(1,y)
    r_index = np.arange(N - 1, 2 * N - 1)
    s_index = np.arange(N - 1, -1, -1)
    Kxy_gain = Krs[r_index, s_index].copy()

    Kxy = None
    if return_Kxy or check:
        Kxy = np.full((N, N), np.nan, dtype=np.float64)
        for i in range(N):
            for j in range(i + 1):
                r_idx = i + j
                s_idx = i - j
                Kxy[i, j] = Krs[r_idx, s_idx]

    if check:
        diag_err = np.nanmax(np.abs(np.diag(Kxy) - F))
        left_boundary_err = np.nanmax(np.abs(Kxy[:, 0]))

        print("solve_k consistency check:")
        print(f"  max diagonal error = {diag_err:.3e}")
        print(f"  max k(x,0) error   = {left_boundary_err:.3e}")

    if return_Kxy:
        return Krs, Kxy, Kxy_gain

    return Krs, Kxy_gain


def solve_g(lambda_func, spatial, check=False):
    """
    Directly solve the observer kernel g(x,y) on 0 <= y <= x <= 1.

    Observer kernel equation:
        g_xx(x,y) - g_yy(x,y) = -lambda(x) g(x,y)

    Boundary conditions:
        g(1,y) = 0
        g(x,x) = -1/2 int_x^1 lambda(s) ds

    Observer gain:
        g1(x) = -g(x,0)

    Return:
        Gxy     : lower-triangular array, Gxy[i,j]=g(x_i,y_j)
        g1_gain : array, g1_gain[i]=g1(x_i)
    """

    spatial = np.asarray(spatial, dtype=float)
    N = len(spatial)
    dx = spatial[1] - spatial[0]

    if not np.allclose(np.diff(spatial), dx):
        raise ValueError("spatial must be a uniform grid.")

    Ns = N
    Nr = 2 * N - 1

    s_grid = np.linspace(0.0, 1.0, Ns)
    r_grid = np.linspace(0.0, 2.0, Nr)
    h = s_grid[1] - s_grid[0]

    ii = np.arange(Nr)[:, None]
    jj = np.arange(Ns)[None, :]
    valid = (ii >= jj) & (ii <= (Nr - 1 - jj))

    Grs = np.full((Nr, Ns), np.nan, dtype=np.float64)

    # Boundary s=0: g(x,x) = -1/2 int_x^1 lambda(z) dz
    x_int = spatial
    lam_int = np.asarray(lambda_func(x_int), dtype=np.float64)

    F = cumulative_trapezoid(lam_int, x_int, initial=0.0)
    F_total = F[-1]

    x_on_r = r_grid / 2.0
    F_on_r = np.interp(x_on_r, x_int, F)

    Grs[:, 0] = -0.5 * (F_total - F_on_r)

    # Boundary x=1: r+s=2
    for j in range(Ns):
        i_right = Nr - 1 - j
        Grs[i_right, j] = 0.0

    # March from right to left
    for j in range(Ns - 1):

        i_right_next = Nr - 1 - (j + 1)

        for i in range(i_right_next - 1, j, -1):

            if not (
                valid[i, j]
                and valid[i + 1, j]
                and valid[i, j + 1]
                and valid[i + 1, j + 1]
            ):
                continue

            if (
                np.isnan(Grs[i + 1, j + 1])
                or np.isnan(Grs[i + 1, j])
                or np.isnan(Grs[i, j])
            ):
                continue

            r_c = 0.5 * (r_grid[i] + r_grid[i + 1])
            s_c = 0.5 * (s_grid[j] + s_grid[j + 1])
            x_c = 0.5 * (r_c + s_c)

            a_c = -0.25 * float(lambda_func(x_c))
            q = (h * h * a_c) / 4.0

            SW = Grs[i, j]
            SE = Grs[i + 1, j]
            NE = Grs[i + 1, j + 1]

            # Solve for NW:
            # NE - SE - NW + SW = h^2 a_c * (SW+SE+NW+NE)/4
            NW_tilde = NE - SE + SW
            NW = (NW_tilde - q * (SW + SE + NE)) / (1.0 + q)

            Grs[i, j + 1] = NW

    # Convert to g(x,y)
    Gxy = np.full((N, N), np.nan, dtype=np.float64)

    for i in range(N):
        for j in range(i + 1):
            r_idx = i + j
            s_idx = i - j
            Gxy[i, j] = Grs[r_idx, s_idx]

    # Observer gain
    g1_gain = -Gxy[:, 0].copy()

    if check:
        diag_target = -0.5 * (F_total - F)
        diag_err = np.nanmax(np.abs(np.diag(Gxy) - diag_target))
        right_err = np.nanmax(np.abs(Gxy[-1, :]))

        print("solve_g consistency check:")
        print(f"  max diagonal error = {diag_err:.3e}")
        print(f"  max g(1,y) error   = {right_err:.3e}")

    return Gxy, g1_gain

# Part3 Build Crank--Nicolson matrices for the interior nodes, and compute the left boundary flux using second-order finite difference.
def build_interior_CN_mats(spatial, dt, lam_profile):
    """
    Interior-node Crank--Nicolson matrices for
        v_t = v_xx + lambda(x) v.
    Boundary values are handled separately.
    """
    spatial = np.asarray(spatial, dtype=float)
    dx = spatial[1] - spatial[0]
    Nx = len(spatial)
    n_int = Nx - 2

    lam_int = lam_profile[1:-1]

    main = -2.0 / dx**2 + lam_int
    off = np.ones(n_int - 1) / dx**2

    A = sp.diags(
        [off, main, off],
        offsets=[-1, 0, 1],
        shape=(n_int, n_int),
        format="csr"
    )

    I = sp.eye(n_int, format="csr")

    M_minus = I - 0.5 * dt * A
    M_plus = I + 0.5 * dt * A

    return M_minus, M_plus


def left_flux_second_order(v, dx):
    """
    u_x(0,t) ≈ (-3u(0,t)+4u(dx,t)-u(2dx,t))/(2dx).
    """
    return (-3.0 * v[0] + 4.0 * v[1] - v[2]) / (2.0 * dx)



In [ ]:
# Part4 Classical and neural closed-loop simulation.

def prepare_outputfeedback_system(lam_param, spatial, dt, check_kernel=False):
    """Compute the gains and CN solvers once for each reaction coefficient."""
    spatial = np.asarray(spatial, dtype=float)
    Nx = len(spatial)
    dx = spatial[1] - spatial[0]

    lambda_func = make_lam_fn(lam_param)
    lam_profile = np.asarray(lambda_func(spatial), dtype=np.float64)

    # Controller kernel boundary gain k(1,y)
    _, k_gain = solve_k(lambda_func, spatial)
    k_gain = np.asarray(k_gain, dtype=np.float64).reshape(-1)

    # Observer gain g1(x)=-g(x,0)
    _, g1_gain = solve_g(lambda_func, spatial, check=check_kernel)
    g1_gain = np.asarray(g1_gain, dtype=np.float64).reshape(-1)

    # Quadrature weights
    wq = get_simpson_coeff(spatial).astype(np.float64)

    # Algebraic boundary control:
    # U = sum_j w_j k_j uhat_j, with uhat_N = U.
    denom = 1.0 - wq[-1] * k_gain[-1]
    if abs(denom) < 1e-10:
        raise RuntimeError("The feedback denominator is too close to zero.")

    cU = (wq[1:-1] * k_gain[1:-1]) / denom

    # Interior CN matrices
    M_minus, M_plus = build_interior_CN_mats(spatial, dt, lam_profile)
    n_int = Nx - 2

    eR = np.zeros(n_int)
    eR[-1] = 1.0

    plant_solver = spla.factorized(M_minus.tocsc())

    # Observer solver with implicit U^{n+1}=cU@uhat^{n+1}
    M_obs = M_minus.tolil()
    M_obs[-1, :] -= (dt / (2.0 * dx**2)) * cU
    obs_solver = spla.factorized(M_obs.tocsc())

    return (lam_profile, wq * k_gain, cU, g1_gain, M_plus,
            eR, plant_solver, obs_solver)


def simulate_outputfeedback_trajectory(
    system,
    spatial,
    nt,
    dt,
    u0_profile,
    uhat0_profile,
):
    """Simulate the classical controller from the supplied compatible initial states."""
    Nx = len(spatial)
    dx = spatial[1] - spatial[0]

    (lam_profile, wk, cU, g1_gain, M_plus,
     eR, plant_solver, obs_solver) = system

    u = np.zeros((nt, Nx), dtype=np.float64)
    uhat = np.zeros((nt, Nx), dtype=np.float64)
    U = np.zeros(nt, dtype=np.float64)
    p = np.zeros(nt, dtype=np.float64)
    phat = np.zeros(nt, dtype=np.float64)

    # Use the same compatible initial conditions as the neural controller
    U[0] = uhat0_profile[-1]
    u[0, :] = u0_profile
    uhat[0, :] = uhat0_profile

    for n in range(nt - 1):
        u_int = u[n, 1:-1]
        uh_int = uhat[n, 1:-1]

        U_old = U[n]

        p[n] = left_flux_second_order(u[n, :], dx)
        phat[n] = left_flux_second_order(uhat[n, :], dx)

        innovation = p[n] - phat[n]

        # Observer update
        rhs_hat = (
            M_plus @ uh_int
            + (dt / (2.0 * dx**2)) * U_old * eR
            + dt * g1_gain[1:-1] * innovation
        )

        uh_new_int = obs_solver(rhs_hat)

        # New control
        U_new = float(cU @ uh_new_int)

        # Plant update
        rhs_u = (
            M_plus @ u_int
            + (dt / (2.0 * dx**2)) * (U_old + U_new) * eR
        )

        u_new_int = plant_solver(rhs_u)

        u[n + 1, 0] = 0.0
        u[n + 1, 1:-1] = u_new_int
        u[n + 1, -1] = U_new

        uhat[n + 1, 0] = 0.0
        uhat[n + 1, 1:-1] = uh_new_int
        uhat[n + 1, -1] = U_new

        U[n + 1] = U_new

    p[-1] = left_flux_second_order(u[-1, :], dx)
    phat[-1] = left_flux_second_order(uhat[-1, :], dx)

    return (
        lam_profile.astype(np.float32),
        p.astype(np.float32),
        U.astype(np.float32),
        u.astype(np.float32),
        uhat.astype(np.float32),
    )



def make_compatible_initial_conditions(spatial, wk, psi, phi):
    U0 = (wk @ phi) / (1.0 - wk @ spatial)
    u0 = psi + spatial * U0
    uhat0 = phi + spatial * U0
    return u0, uhat0, U0


def simulate_open_loop(lam_profile, spatial, nt, dt, u0_profile):
    """Switch the boundary input to zero at t=0+; keep the initial interior."""
    M_minus, M_plus = build_interior_CN_mats(spatial, dt, lam_profile)
    plant_solver = spla.factorized(M_minus.tocsc())

    u = np.zeros((nt, len(spatial)), dtype=np.float64)
    u[0, 1:-1] = u0_profile[1:-1]

    for n in range(nt - 1):
        u[n + 1, 1:-1] = plant_solver(M_plus @ u[n, 1:-1])

    return u


@torch.no_grad()
def run_closed_loop_neural_controller(
    model,
    spatial,
    stats,
    lam_param,
    nt,
    dt,
    u0_profile,
    uhat0_profile,
):
    """Causal sampled-data control with zero-order hold on each time interval."""
    model.eval()
    model_device = next(model.parameters()).device
    dx = spatial[1] - spatial[0]
    Nx = len(spatial)
    lam_profile = make_lam_fn(lam_param)(spatial)

    M_minus, M_plus = build_interior_CN_mats(spatial, dt, lam_profile)
    plant_solver = spla.factorized(M_minus.tocsc())
    eR = np.zeros(Nx - 2)
    eR[-1] = 1.0

    # Compute the two static embeddings once.
    lam_norm = (lam_profile.astype(np.float32) - stats["lam_mean"]) / stats["lam_std"]
    uhat0_norm = uhat0_profile.astype(np.float32) / stats["uhat0_std"]

    lam_tensor = torch.from_numpy(lam_norm[None, :]).to(model_device)
    uhat0_tensor = torch.from_numpy(uhat0_norm[None, :]).to(model_device)

    z_lambda, spatial_feature = model.lambda_encoder(lam_tensor)
    z_0 = model.ic_encoder(model.wq * spatial_feature * uhat0_tensor)
    z_f = torch.cat([z_lambda, z_0], dim=-1)

    u = np.zeros((nt, Nx), dtype=np.float64)
    p = np.zeros(nt, dtype=np.float64)
    U = np.zeros(nt, dtype=np.float64)
    U_pred = np.zeros(nt, dtype=np.float64)
    u[0, :] = u0_profile

    # Exact compatible boundary at the initial time.
    U0 = float(uhat0_profile[-1])
    state = None

    for n in range(nt):
        p[n] = left_flux_second_order(u[n, :], dx)
        p_norm = (p[n] - stats["p_mean"]) / stats["p_std"]
        p_tensor = z_f.new_tensor([[p_norm]])

        lstm_input = torch.cat([z_f, p_tensor], dim=-1).unsqueeze(1)
        lstm_out, state = model.lstm(lstm_input, state)
        U_norm = model.head(lstm_out).item()
        U_pred[n] = U_norm * float(stats["U_std"]) + float(stats["U_mean"])

        # Record the raw network prediction separately from the applied input.
        # At t=0 use the known compatible boundary; after that use the network.
        U[n] = U0 if n == 0 else U_pred[n]
        u[n, -1] = U[n]

        if n < nt - 1:
            # U[n] is held on [t_n, t_{n+1}); both CN boundary terms use it.
            rhs = M_plus @ u[n, 1:-1] + (dt / dx**2) * U[n] * eR
            u[n + 1, 1:-1] = plant_solver(rhs)
            u[n + 1, -1] = U[n]

    return {
        "time": np.arange(nt) * dt,
        "u": u,
        "p": p,
        "U": U,
        "U_pred": U_pred,
    }

In [ ]:
# Part9 Define the DeepONet-LSTM controller with observer initial condition.
import os
os.environ["DDE_BACKEND"] = "pytorch"

import deepxde as dde
import torch
import torch.nn as nn
import numpy as np


class LambdaEncoder(nn.Module):
    """
    DeepXDE-based DeepONet encoder.

    Input:
        lam_profile: [B, Nx]

    Output:
        z_lambda:        [B, emb_dim]
        spatial_feature: [B, Nx]

    The DeepONetCartesianProd maps
        lambda(x) -> a spatial feature on the fixed grid.
    Then an MLP compresses this spatial feature into z_lambda.
    """

    def __init__(
        self,
        spatial,
        emb_dim=32,
        feature_dim=128,
        branch_hidden=(256, 256),
        trunk_hidden=(128, 256),
        activation="relu",
        kernel="Glorot normal",
    ):
        super().__init__()

        spatial = np.asarray(spatial, dtype=np.float32)
        self.nx = len(spatial)

        self.register_buffer(
            "grid",
            torch.from_numpy(spatial.reshape(-1, 1)).float()
        )

        branch_layers = [self.nx, *branch_hidden, feature_dim]
        trunk_layers = [1, *trunk_hidden, feature_dim]

        self.deeponet = dde.nn.DeepONetCartesianProd(
            branch_layers,
            trunk_layers,
            activation,
            kernel,
        )

        self.embed = nn.Sequential(
            nn.Linear(self.nx, 128),
            nn.ReLU(),
            nn.Linear(128, emb_dim),
            nn.ReLU(),
        )

    def forward(self, lam_profile):
        spatial_feature = self.deeponet((lam_profile, self.grid))
        z_lambda = self.embed(spatial_feature)
        return z_lambda, spatial_feature


class LambdaLSTMController(nn.Module):
    """
    Hybrid DeepONet-LSTM controller with observer initial condition.

    Input:
        lam_profile:   [B, Nx]
        uhat0_profile: [B, Nx]
        p_seq:         [B, T], where p(t)=u_x(0,t)

    Output:
        U_seq      : [B, T]
    """

    def __init__(
        self,
        spatial,
        emb_dim=32,
        lstm_hidden=64,
        num_layers=1,
        feature_dim=128,
        branch_hidden=(256, 256),
        trunk_hidden=(128, 256),
        activation="relu",
        kernel="Glorot normal",
    ):
        super().__init__()

        self.lambda_encoder = LambdaEncoder(
            spatial=spatial,
            emb_dim=emb_dim,
            feature_dim=feature_dim,
            branch_hidden=branch_hidden,
            trunk_hidden=trunk_hidden,
            activation=activation,
            kernel=kernel,
        )

        self.register_buffer(
            "wq",
            torch.as_tensor(get_simpson_coeff(spatial), dtype=torch.float32)
        )

        self.ic_encoder = nn.Linear(len(spatial), emb_dim, bias=False)

        self.lstm = nn.LSTM(
            input_size=2 * emb_dim + 1,
            hidden_size=lstm_hidden,
            num_layers=num_layers,
            batch_first=True,
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, lam_profile, uhat0_profile, p_seq):
        _, T = p_seq.shape

        z_lambda, spatial_feature = self.lambda_encoder(lam_profile)

        # Integral feature encoding
        z_0 = self.ic_encoder(self.wq * spatial_feature * uhat0_profile)

        # Feature concatenation
        z_f = torch.cat([z_lambda, z_0], dim=-1)
        z_seq = z_f.unsqueeze(1).repeat(1, T, 1)

        lstm_input = torch.cat([z_seq, p_seq.unsqueeze(-1)], dim=-1)
        lstm_out, _ = self.lstm(lstm_input)

        U_seq = self.head(lstm_out).squeeze(-1)
        return U_seq

In [ ]:
def load_trained_controller(model_path):
    checkpoint = torch.load(
        model_path,
        map_location=device,
        weights_only=False
    )

    spatial_loaded = checkpoint["spatial"]
    stats_loaded = checkpoint["stats"]
    config = checkpoint["config"]

    model_loaded = LambdaLSTMController(
        spatial=spatial_loaded,
        emb_dim=config["emb_dim"],
        lstm_hidden=config["hidden_size"],
        num_layers=config["num_layers"],
        feature_dim=config["feature_dim"],
        branch_hidden=tuple(config.get("branch_hidden", (256, 256))),
        trunk_hidden=tuple(config.get("trunk_hidden", (128, 256))),
        activation=config.get("activation", "relu"),
        kernel=config.get("kernel", "Glorot normal"),
    ).to(device)

    model_loaded.load_state_dict(checkpoint["state_dict"])
    model_loaded.eval()

    print("Loaded model from:", model_path)
    print("Best val loss:", checkpoint.get("best_val_loss", None))

    return model_loaded, spatial_loaded, stats_loaded

In [ ]:
# Load the new three-input MSE model.
# Replace this path with a specific checkpoint if needed.
model_path = sorted(Path("./runs").glob(
    "lambda_lstm_controller_uhat0_mse_*/best_model.pth"
))[-1]

model, spatial_model, stats = load_trained_controller(model_path)

In [ ]:
# Closed-loop comparison from the same compatible initial conditions.
beta_test = 6.5
spatial = np.linspace(0.0, 1.0, len(spatial_model), dtype=np.float64)
dt = 0.001
nt = 2000
time_grid = np.arange(nt) * dt

rng = np.random.default_rng(2027)
psi = make_initial_profile(
    spatial, init_type="cheb_random", amp=rng.uniform(0.2, 1.0), rng=rng,
    n_modes=8, decay_power=2.0, normalize=True,
)
phi = make_initial_profile(
    spatial, init_type="cheb_random", amp=rng.uniform(0.2, 1.0), rng=rng,
    n_modes=8, decay_power=2.0, normalize=True,
)

system = prepare_outputfeedback_system(beta_test, spatial, dt)
lam_profile, wk = system[:2]
u0_profile, uhat0_profile, U0 = make_compatible_initial_conditions(
    spatial, wk, psi, phi
)

u_open = simulate_open_loop(lam_profile, spatial, nt, dt, u0_profile)

lam_profile_bs, p_bs, U_bs, u_bs, uhat_bs = simulate_outputfeedback_trajectory(
    system, spatial, nt, dt, u0_profile, uhat0_profile
)

res_nn = run_closed_loop_neural_controller(
    model=model,
    spatial=spatial,
    stats=stats,
    lam_param=beta_test,
    nt=nt,
    dt=dt,
    u0_profile=u0_profile,
    uhat0_profile=uhat0_profile,
)
u_nn = res_nn["u"]
U_nn = res_nn["U"]
U_pred_nn = res_nn["U_pred"]
p_nn = res_nn["p"]

u_diff = u_bs - u_nn
U_diff = U_bs - U_nn

print("Initial control:", U0)
print("Initial neural prediction:", U_pred_nn[0])
print("Initial prediction error:", abs(U_pred_nn[0] - U0))

In [ ]:
def compute_l2_time_history(u, spatial):
    wq = get_simpson_coeff(spatial)
    return np.sqrt((u.astype(np.float64) ** 2) @ wq)


def relative_l2(pred, ref, weights):
    return np.sqrt(np.sum((pred - ref) ** 2 * weights) /
                   np.sum(ref ** 2 * weights))


l2_open = compute_l2_time_history(u_open, spatial)
l2_bs = compute_l2_time_history(u_bs, spatial)
l2_nn = compute_l2_time_history(u_nn, spatial)
l2_uhat_bs = compute_l2_time_history(uhat_bs, spatial)
l2_error_bs = compute_l2_time_history(u_bs - uhat_bs, spatial)

wq = get_simpson_coeff(spatial)
wt = get_simpson_coeff(time_grid)
U_rel_l2 = relative_l2(U_nn, U_bs, wt)
u_rel_l2 = relative_l2(u_nn, u_bs, wt[:, None] * wq[None, :])

# Include the raw prediction at t=0 when measuring the network's early error.
early = time_grid <= 0.1
U_early_rel_l2 = relative_l2(
    U_pred_nn[early], U_bs[early], get_simpson_coeff(time_grid[early])
)

print("===== Closed-loop comparison =====")
print(f"beta: {beta_test}")
print(f"Final plant L2 (open): {l2_open[-1]:.4e}")
print(f"Final plant L2 (BS): {l2_bs[-1]:.4e}")
print(f"Final plant L2 (NN): {l2_nn[-1]:.4e}")
print(f"Final observer error L2 (BS): {l2_error_bs[-1]:.4e}")
print(f"Max plant L2 (NN): {np.max(l2_nn):.4e}")
print(f"Applied U relative L2 error: {U_rel_l2:.4e}")
print(f"State relative L2 error: {u_rel_l2:.4e}")
print(f"Raw neural U relative L2 error [0, 0.1]: {U_early_rel_l2:.4e}")
print(f"Initial raw U absolute error: {abs(U_pred_nn[0] - U0):.4e}")

In [ ]:
# Initial conditions and open-loop response.
plot_2d_draw_style(
    x=spatial, ys=[u0_profile, uhat0_profile],
    labels=[r"$u_0$", r"$\hat u_0$"], linestyles=["-", "--"],
    xlabel=r"$x$", ylabel="Initial conditions",
    filename=fig_dir / "initial_conditions.jpg",
)

n_plot = 1000
plot_3d_response_draw_style(
    u_data=u_open[:n_plot], spatial=spatial, temporal=time_grid[:n_plot],
    zlabel=r"$u$", filename=fig_dir / "u_open.jpg",
    rstride=30, cstride=4,
)

In [ ]:
# Closed-loop state responses.
for response, label, name in [
    (u_bs, r"$u_{\rm bs}$", "u_backstepping.jpg"),
    (u_nn, r"$u_{\rm N}$", "u_neural.jpg"),
    (u_diff, r"$u_{\rm bs}-u_{\rm N}$", "difference_bs_nn.jpg"),
]:
    plot_3d_response_draw_style(
        u_data=response, spatial=spatial, temporal=time_grid,
        zlabel=label, filename=fig_dir / name,
        rstride=30, cstride=4,
    )

In [ ]:
# Compare the applied boundary inputs and the closed-loop state norms.
plot_2d_draw_style_with_inset(
    x=time_grid, ys=[l2_bs, l2_nn],
    labels=["Backstepping", "Neural controller"],
    linestyles=["-", "--"],
    xlabel=r"$t$", ylabel=r"$\|u(\cdot,t)\|_{L^2}$",
    zoom_xlim=(0.3, 0.8), inset_bbox=(0.4, 0.28, 0.36, 0.32),
    filename=fig_dir / "comp_L2.jpg",
)

plot_2d_draw_style_with_inset(
    x=time_grid, ys=[U_bs, U_nn],
    labels=["Backstepping", "Neural controller"],
    linestyles=["-", "--"],
    xlabel=r"$t$", ylabel="Control input",
    zoom_xlim=(0.0, 0.1), inset_bbox=(0.4, 0.5, 0.36, 0.32),
    filename=fig_dir / "comp_U.jpg",
)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")
print(model)